# Brain Age Prediction: L'Analisi Definitiva (Multimodale)
Questo notebook carica **tutti e 10 i modelli** (5 FLAIR + 5 T1) e calcola in un colpo solo tutte le metriche sul Test Set per permetterti di confrontare:
1. Miglior Modello Singolo T1
2. K-Fold Ensemble T1 (Media di 5 reti)
3. Miglior Modello Singolo FLAIR
4. K-Fold Ensemble FLAIR (Media di 5 reti)
5. Hybrid Ensemble Avanzato: (1/2 Miglior FLAIR + 1/2 Ensemble T1)
6. **Simple Hybrid Ensemble**: (1/2 Miglior FLAIR + 1/2 Miglior T1)

Al termine, identificherà automaticamente il modello col **MAE Assoluto più basso** e traccerà uno Scatter Plot esclusivo per quest'ultimo.

In [ ]:
!rm -rf SFCN
!git clone https://github.com/PietroSchgor/SFCN.git

import sys
sys.path.append('/kaggle/working/SFCN')

In [ ]:
import os
import json
import glob
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

from dp_model.model_files.sfcn import SFCN
from dp_model import dp_utils as dpu

PLOTS_DIR = '/kaggle/working/plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

## 1. Dataset Custom (Multimodale)

In [ ]:
class BrainAgeDataset(Dataset):
    def __init__(self, data_dir, modality='FLAIR', is_train=False):
        self.data_dir = data_dir
        self.modality = modality 
        self.is_train = is_train
        self.subject_dirs = sorted(glob.glob(os.path.join(data_dir, "sub-*")))
        self.samples = []
        
        self.bin_range = [0, 70]
        self.bin_step = 1
        self.sigma = 1.0
        
        for subj_dir in self.subject_dirs:
            subj_id = os.path.basename(subj_dir)
            
            nii_path = os.path.join(subj_dir, f"{subj_id}_{self.modality}_MNI152_1mm.nii")
            if not os.path.exists(nii_path):
                nii_path = nii_path + ".gz"
                if not os.path.exists(nii_path):
                    if self.modality == 'T1w':
                        nii_path_alt = os.path.join(subj_dir, f"{subj_id}_T1_MNI152_1mm.nii.gz")
                        if os.path.exists(nii_path_alt):
                            nii_path = nii_path_alt
                        else:
                            continue
                    else:
                        continue
                    
            json_path = os.path.join(subj_dir, f"{subj_id}_participant_info.json")
            if not os.path.exists(json_path):
                continue
                
            with open(json_path, 'r') as f:
                info = json.load(f)
                
            participant_info = info.get("participant_info", {})
            age_cat_val = participant_info.get("age_scan")
            
            if age_cat_val is None:
                continue
            
            try:
                age_cat = int(age_cat_val) - 1
                true_age = 3 + age_cat * 5
                y, _ = dpu.num2vect(true_age, self.bin_range, self.bin_step, self.sigma)
            except:
                continue
            
            self.samples.append({
                "nii_path": nii_path,
                "label_vect": y,
                "true_age": true_age
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img = nib.load(sample['nii_path'])
        data = img.get_fdata(dtype=np.float32)
        
        mean_val = np.mean(data)
        if mean_val > 0:
            data = data / mean_val
            
        in_sp = data.shape
        out_sp = (160, 192, 160)
        
        dx, dy, dz = 0, 0, 0
        
        x_c = int((in_sp[0] - out_sp[0]) / 2) + dx
        y_c = int((in_sp[1] - out_sp[1]) / 2) + dy
        z_c = int((in_sp[2] - out_sp[2]) / 2) + dz
        
        data = data[x_c:x_c+out_sp[0], y_c:y_c+out_sp[1], z_c:z_c+out_sp[2]]
        data = np.expand_dims(data, axis=0)
        
        tensor_data = torch.from_numpy(data)
        label_vect = torch.tensor(sample['label_vect'], dtype=torch.float32)
        
        return tensor_data, label_vect, sample['true_age']

## 2. Configurazione e Caricamento dei 10 Modelli

In [ ]:
KAGGLE_DATA_DIR = "/kaggle/input/datasets/elenaschgor/dataset-2-t1-flair/ds004199_final/"

# --- CONFIGURAZIONE PATH E INDICI MIGLIORI --- 
PATHS_5_FOLD_FLAIR = [
    "/kaggle/input/sfcn-pesi/sfcn_FLAIR_fold_1.pth",
    "/kaggle/input/sfcn-pesi/sfcn_FLAIR_fold_2.pth",
    "/kaggle/input/sfcn-pesi/sfcn_FLAIR_fold_3.pth",
    "/kaggle/input/sfcn-pesi/sfcn_FLAIR_fold_4.pth",
    "/kaggle/input/sfcn-pesi/sfcn_FLAIR_fold_5.pth"
]
IDX_MIGLIOR_FLAIR = 0 

PATHS_5_FOLD_T1 = [
    "/kaggle/input/sfcn-pesi/sfcn_T1w_fold_1.pth",
    "/kaggle/input/sfcn-pesi/sfcn_T1w_fold_2.pth",
    "/kaggle/input/sfcn-pesi/sfcn_T1w_fold_3.pth",
    "/kaggle/input/sfcn-pesi/sfcn_T1w_fold_4.pth",
    "/kaggle/input/sfcn-pesi/sfcn_T1w_fold_5.pth"
]
IDX_MIGLIOR_T1 = 0 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dummy_dataset = BrainAgeDataset(KAGGLE_DATA_DIR, modality='FLAIR', is_train=False)
dataset_size = len(dummy_dataset)

if dataset_size > 0:
    all_ages = [sample['true_age'] for sample in dummy_dataset.samples]
    all_indices = np.arange(dataset_size)
    _, test_idx = train_test_split(all_indices, test_size=0.10, random_state=42, stratify=all_ages)
    
    test_dataset_flair = torch.utils.data.Subset(BrainAgeDataset(KAGGLE_DATA_DIR, modality='FLAIR', is_train=False), test_idx)
    test_loader_flair = DataLoader(test_dataset_flair, batch_size=1, shuffle=False)
    
    test_dataset_t1 = torch.utils.data.Subset(BrainAgeDataset(KAGGLE_DATA_DIR, modality='T1w', is_train=False), test_idx)
    test_loader_t1 = DataLoader(test_dataset_t1, batch_size=1, shuffle=False)
    
    print("Caricamento delle 5 Reti FLAIR...")
    models_flair = []
    for p in PATHS_5_FOLD_FLAIR:
        m = SFCN(output_dim=70)
        m.load_state_dict(torch.load(p, map_location=device))
        m.to(device)
        m.eval()
        models_flair.append(m)
        
    print("Caricamento delle 5 Reti T1...")
    models_t1 = []
    for p in PATHS_5_FOLD_T1:
        m = SFCN(output_dim=70)
        m.load_state_dict(torch.load(p, map_location=device))
        m.to(device)
        m.eval()
        models_t1.append(m)

## 3. Calcolo Combinato e Generazione Plot

In [ ]:
if dataset_size > 0:
    print("\n==============================================")
    print("   CALCOLO PREDITTIVO MULTIPLO SUL TEST SET")
    print("==============================================")
    
    all_true_ages = []
    
    preds_t1_best = []
    preds_t1_ensemble = []
    preds_flair_best = []
    preds_flair_ensemble = []
    preds_hybrid = []
    preds_simple_hybrid = []
    
    err_t1_best = []
    err_t1_ensemble = []
    err_flair_best = []
    err_flair_ensemble = []
    err_hybrid = []
    err_simple_hybrid = []
    
    bin_centers = np.arange(0, 70, 1)
    
    with torch.no_grad():
        for (inputs_flair, _, true_age), (inputs_t1, _, _) in zip(test_loader_flair, test_loader_t1):
            inputs_flair, inputs_t1 = inputs_flair.to(device), inputs_t1.to(device)
            true_age_val = true_age.item()
            
            # --- ESTRAZIONE PROBABILITÀ FLAIR ---
            probs_flair_list = []
            for m in models_flair:
                out = m(inputs_flair)[0].view(1, -1)
                probs_flair_list.append(torch.exp(out).cpu().numpy())
                
            prob_flair_best = probs_flair_list[IDX_MIGLIOR_FLAIR]
            prob_flair_ensemble = np.mean(probs_flair_list, axis=0)
            
            # --- ESTRAZIONE PROBABILITÀ T1 ---
            probs_t1_list = []
            for m in models_t1:
                out = m(inputs_t1)[0].view(1, -1)
                probs_t1_list.append(torch.exp(out).cpu().numpy())
                
            prob_t1_best = probs_t1_list[IDX_MIGLIOR_T1]
            prob_t1_ensemble = np.mean(probs_t1_list, axis=0)
            
            # --- HYBRID ENSEMBLE (Miglior FLAIR + K-Fold T1) ---
            prob_hybrid = (0.5 * prob_flair_best) + (0.5 * prob_t1_ensemble)
            
            # --- SIMPLE HYBRID ENSEMBLE (Miglior FLAIR + Miglior T1) ---
            prob_simple_hybrid = (0.5 * prob_flair_best) + (0.5 * prob_t1_best)
            
            # --- CONVERSIONE IN ANNI ---
            age_t1_best = (prob_t1_best @ bin_centers)[0]
            age_t1_ens = (prob_t1_ensemble @ bin_centers)[0]
            age_flair_best = (prob_flair_best @ bin_centers)[0]
            age_flair_ens = (prob_flair_ensemble @ bin_centers)[0]
            age_hybrid = (prob_hybrid @ bin_centers)[0]
            age_simple_hybrid = (prob_simple_hybrid @ bin_centers)[0]
            
            # Salvataggio Predizioni per lo Scatter Plot
            all_true_ages.append(true_age_val)
            preds_t1_best.append(age_t1_best)
            preds_t1_ensemble.append(age_t1_ens)
            preds_flair_best.append(age_flair_best)
            preds_flair_ensemble.append(age_flair_ens)
            preds_hybrid.append(age_hybrid)
            preds_simple_hybrid.append(age_simple_hybrid)
            
            # Calcolo Errori
            err_t1_best.append(abs(age_t1_best - true_age_val))
            err_t1_ensemble.append(abs(age_t1_ens - true_age_val))
            err_flair_best.append(abs(age_flair_best - true_age_val))
            err_flair_ensemble.append(abs(age_flair_ens - true_age_val))
            err_hybrid.append(abs(age_hybrid - true_age_val))
            err_simple_hybrid.append(abs(age_simple_hybrid - true_age_val))
            
    # --- STAMPA RISULTATI ---
    print("\n==============================================")
    print("        RISULTATI UFFICIALI SUL TEST SET")
    print("==============================================")
    print(f"1) Miglior Singolo Modello T1      : {np.mean(err_t1_best):.3f} anni")
    print(f"2) K-Fold Ensemble T1 (5 Reti)     : {np.mean(err_t1_ensemble):.3f} anni")
    print("----------------------------------------------")
    print(f"3) Miglior Singolo Modello FLAIR   : {np.mean(err_flair_best):.3f} anni")
    print(f"4) K-Fold Ensemble FLAIR (5 Reti)  : {np.mean(err_flair_ensemble):.3f} anni")
    print("==============================================")
    print(f"5) HYBRID ENSEMBLE AVANZATO        : {np.mean(err_hybrid):.3f} anni")
    print("   (50% Best FLAIR + 50% Ensemble T1)")
    print("----------------------------------------------")
    print(f"6) SIMPLE HYBRID ENSEMBLE          : {np.mean(err_simple_hybrid):.3f} anni")
    print("   (50% Best FLAIR + 50% Best T1)")
    print("==============================================\n")
    
    # --- IDENTIFICAZIONE DEL VINCITORE ASSOLUTO ---
    # Creiamo un dizionario con i MAE medi e un dizionario con le rispettive predizioni
    mae_dict = {
        'Miglior Singolo T1': np.mean(err_t1_best),
        'K-Fold Ensemble T1': np.mean(err_t1_ensemble),
        'Miglior Singolo FLAIR': np.mean(err_flair_best),
        'K-Fold Ensemble FLAIR': np.mean(err_flair_ensemble),
        'Hybrid Ensemble Avanzato (1 FLAIR + 5 T1)': np.mean(err_hybrid),
        'Simple Hybrid Ensemble (1 FLAIR + 1 T1)': np.mean(err_simple_hybrid)
    }
    
    pred_dict = {
        'Miglior Singolo T1': preds_t1_best,
        'K-Fold Ensemble T1': preds_t1_ensemble,
        'Miglior Singolo FLAIR': preds_flair_best,
        'K-Fold Ensemble FLAIR': preds_flair_ensemble,
        'Hybrid Ensemble Avanzato (1 FLAIR + 5 T1)': preds_hybrid,
        'Simple Hybrid Ensemble (1 FLAIR + 1 T1)': preds_simple_hybrid
    }
    
    best_model_name = min(mae_dict, key=mae_dict.get)
    best_model_mae = mae_dict[best_model_name]
    winning_preds = pred_dict[best_model_name]
    
    print(f"\n>>> Il Modello Vincitore Assoluto è: {best_model_name} (MAE: {best_model_mae:.3f})")
    
    # --- PLOT: SCATTER PLOT DEL SOLO VINCITORE ASSOLUTO ---
    plt.figure(figsize=(9, 7))
    plt.scatter(all_true_ages, winning_preds, color='forestgreen', edgecolor='black', alpha=0.8, s=80)
    
    min_val = min(min(all_true_ages), min(winning_preds)) - 2
    max_val = max(max(all_true_ages), max(winning_preds)) + 2
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2.5, label='Predizione Perfetta')
    
    plt.title(f'Vincitore Assoluto: {best_model_name}\nMAE: {best_model_mae:.3f} Anni', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Età Reale (Anni)', fontsize=12)
    plt.ylabel('Età Predetta (Anni)', fontsize=12)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, '07_absolute_winner_scatter.png'), dpi=300)
    plt.show()
